# Phase B: 注入时序 + 协同过滤信号

## 目标
在 Phase A-3 的合成交互数据基础上：
1. 注入时间戳（模拟 2026-03-01 ~ 2026-08-11，约 5.5 个月）
2. 热门 item 幂律分布（α=1.3，头部效应）
3. 全局共享池（80 items，40% 曝光 — 保证基础 Jaccard > 5%）
4. 同簇用户共享 item（25 簇，每簇 250 items 池，25% 曝光 — 协同过滤信号）
5. 用户 session 切分（3-5 个 session/用户）
6. 验证：user-user Jaccard overlap > 5%，同簇提升 > 1.15×

## 输入
- data/user_latent.csv — 用户画像
- data/item_latent.csv — 物品画像
- data/interactions_synthetic.csv — Phase A-3 合成交互

## 输出
- data/interactions_enriched.csv — 带时间戳 + CF 信号的交互数据

In [ ]:
import pandas as pd, numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize
from scipy.special import expit as sigmoid
from collections import defaultdict
import matplotlib.pyplot as plt
import json, warnings
warnings.filterwarnings('ignore')

DATA_DIR = '../data'
SEED = 42
np.random.seed(SEED)
rng = np.random.RandomState(SEED)
print('Setup OK')

In [ ]:
# 加载数据
user_latent = pd.read_csv(f'{DATA_DIR}/user_latent.csv', index_col='user_id')
item_latent = pd.read_csv(f'{DATA_DIR}/item_latent.csv', index_col='item_id')
interactions = pd.read_csv(f'{DATA_DIR}/interactions_synthetic.csv')
with open(f'{DATA_DIR}/match_weights.json') as f:
    BEST_WEIGHTS = json.load(f)

# Fix double prefix
rename_map = {}
for c in user_latent.columns:
    if c.startswith('theme_theme_'):
        rename_map[c] = c.replace('theme_theme_', 'theme_')
user_latent.rename(columns=rename_map, inplace=True)

print(f'Users: {len(user_latent):,}, Items: {len(item_latent):,}, Interactions: {len(interactions):,}')
iu = interactions.groupby('user_id').size()
print(f'Interactions per user (original): mean={iu.mean():.1f}, median={iu.median():.0f}')

---
## 1. 热门 Item 幂律分布

用 Pareto 分布模拟头部效应：top 5% item 获得 ~40% 曝光

In [ ]:
n_items = len(item_latent)
alpha = 1.3
ranks = np.arange(1, n_items + 1)
item_weights = ranks ** (-alpha)
item_probs = item_weights / item_weights.sum()

top5_n = int(n_items * 0.05)
top5_share = item_probs[:top5_n].sum()
print(f'Top 5% items ({top5_n:,}): {top5_share*100:.1f}% of total weight')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].loglog(ranks, item_weights, linewidth=1, color='#4ECDC4')
axes[0].set_xlabel('Item Rank'); axes[0].set_ylabel('Weight')
axes[0].set_title(f'Power-law Item Popularity (alpha={alpha})')
axes[1].plot(np.arange(1, n_items+1), item_probs.cumsum(), linewidth=1.5, color='#FF6B6B')
axes[1].axhline(y=0.5, color='#333', linestyle='--', alpha=0.5)
axes[1].axvline(x=top5_n, color='#333', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Item Rank'); axes[1].set_ylabel('Cumulative')
axes[1].annotate(f'Top 5%: {top5_share*100:.0f}%', xy=(top5_n, 0.5), fontsize=10)
fig.tight_layout(); plt.show()

---
## 2. 用户 Theme 聚类

按 theme_affinity (SVD 50 维) 聚成 25 个簇，同簇用户共享 item

In [ ]:
THEME_COLS = [c for c in user_latent.columns if c.startswith('theme_')]
user_theme_vecs = user_latent[THEME_COLS].values
user_theme_norm = normalize(user_theme_vecs, norm='l2')

N_CLUSTERS = 25
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init=10)
user_clusters = kmeans.fit_predict(user_theme_norm)
user_latent['cluster'] = user_clusters

print(f'Clusters: {N_CLUSTERS}, sizes: {np.bincount(user_clusters).min()}-{np.bincount(user_clusters).max()}')

# 每簇共享 item 池（小而集中，增强簇内重叠）
CLUSTER_SHARE_SIZE = 250
item_ids_arr = item_latent.index.values
cluster_item_pool = {}
for c in range(N_CLUSTERS):
    cluster_item_pool[c] = set(rng.choice(item_ids_arr, size=CLUSTER_SHARE_SIZE, replace=False))

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(user_theme_vecs[:, 0], user_theme_vecs[:, 1],
                     c=user_clusters, cmap='tab20', alpha=0.5, s=5)
ax.set_xlabel('Theme SVD 1'); ax.set_ylabel('Theme SVD 2')
ax.set_title(f'User Theme Clusters (KMeans, k={N_CLUSTERS})')
plt.colorbar(scatter, ax=ax, label='Cluster')
fig.tight_layout(); plt.show()

---
## 3. 注入时间戳

模拟 2026-03-01 ~ 2026-08-11。每用户 3-5 个 session，傍晚高峰，session 内密集

In [ ]:
TIME_START = np.datetime64('2026-03-01')
TIME_END = np.datetime64('2026-08-11')
TOTAL_DAYS = (TIME_END - TIME_START).astype(int)
print(f'Time range: {TIME_START} ~ {TIME_END} ({TOTAL_DAYS} days)')

HOUR_PROBS = np.ones(24)
HOUR_PROBS[18:24] = 3.0; HOUR_PROBS[8:12] = 1.5; HOUR_PROBS[12:18] = 2.0
HOUR_PROBS = HOUR_PROBS / HOUR_PROBS.sum()

user_sessions = {}
for uid in user_latent.index:
    n_sessions = rng.randint(3, 6)
    session_days = np.sort(rng.choice(TOTAL_DAYS, size=n_sessions, replace=False))
    events_per_session = rng.randint(15, 26, size=n_sessions)
    sessions = []
    for day, n_evt in zip(session_days, events_per_session):
        hour = rng.choice(24, p=HOUR_PROBS)
        minute = rng.randint(0, 60)
        base = TIME_START + np.timedelta64(int(day), 'D') + np.timedelta64(hour, 'h') + np.timedelta64(minute, 'm')
        sessions.append((base, n_evt))
    user_sessions[uid] = sessions

n_sess = [len(v) for v in user_sessions.values()]
n_evt = [sum(s[1] for s in v) for v in user_sessions.values()]
print(f'Sessions/user: mean={np.mean(n_sess):.1f}')
print(f'Events/user: mean={np.mean(n_evt):.1f}, total={sum(n_evt):,}')

---
## 4. 预计算 Match Score 组件

In [ ]:
SKILL_COLS = [c for c in item_latent.columns if c.startswith('skill_')]
CT_COLS = ['ct_published_asset', 'ct_published_prompt', 'ct_published_commerce']

item_theme_vecs = item_latent[THEME_COLS].values
item_skill_vecs = item_latent[SKILL_COLS].values
item_ct_onehot = item_latent[CT_COLS].values
item_quality = item_latent['quality_score'].values
item_price = item_latent['price_tier'].values
item_freshness = item_latent['freshness'].values
item_popularity = item_latent['popularity'].values
item_idx_map = {iid: i for i, iid in enumerate(item_ids_arr)}
item_is_commerce = item_latent['ct_published_commerce'].values.astype(bool)

def compute_raw(user_vec, item_idx):
    """Return raw score (before sigmoid)."""
    u_theme = user_vec[THEME_COLS].values.astype(float)
    u_skill = user_vec[SKILL_COLS].values.astype(float)
    u_ct = user_vec[CT_COLS].values.astype(float)
    qsens = float(user_vec['quality_sensitivity'])
    budget = float(user_vec['budget_level'])
    it, isk = item_theme_vecs[item_idx], item_skill_vecs[item_idx]
    ut_norm = np.linalg.norm(u_theme)
    t = 0.0 if ut_norm < 1e-12 else float(np.dot(it/(np.linalg.norm(it)+1e-12), u_theme/ut_norm))
    p = 1.0 - abs(budget - item_price[item_idx])
    us_norm = np.linalg.norm(u_skill)
    s = 0.0 if us_norm < 1e-12 else float(np.dot(isk/(np.linalg.norm(isk)+1e-12), u_skill/us_norm))
    q = qsens * item_quality[item_idx]
    ct = float(np.dot(item_ct_onehot[item_idx], u_ct))
    raw = (BEST_WEIGHTS['theme']*t + BEST_WEIGHTS['price']*p + BEST_WEIGHTS['skill']*s +
           BEST_WEIGHTS['quality']*q + BEST_WEIGHTS['ct']*ct +
           BEST_WEIGHTS['pop']*item_popularity[item_idx] -
           BEST_WEIGHTS['fresh']*(1.0-item_freshness[item_idx]))
    return float(raw)

def compute_score(user_vec, item_idx):
    """Return match_score = sigmoid(K * raw). K=8.0 amplifies dynamic range ~4x."""
    return max(0.0, min(1.0, float(sigmoid(8.0 * compute_raw(user_vec, item_idx)))))

# ============================================================
# Plan C: Behavior-specific per-item shifts (A_COMMON approach)
# ============================================================
# All behaviors share the SAME raw score from compute_raw().
# Each behavior adds a per-item shift to create partial independence
# while maintaining natural correlation through the shared base score.
#
# A_COMMON = 8.0 (same K as match_score, keeps consistent dynamic range)
# Shift magnitudes tuned to balance:
#   - Shared signal (correlation 0.7-0.98 at probability level)
#   - Per-behavior differentiation (correlation 0.01-0.11 at binary level)
#   - Target behavior rates

A_COMMON = 8.0
item_shift_like    = np.zeros(n_items)                                # reference: no shift
item_shift_long    = 0.12 * item_ct_onehot[:, 0].astype(float)       # assets → longer viewing
item_shift_fav     = 0.08 * item_quality                             # high quality → more collections
item_shift_comment = 0.15 * item_ct_onehot[:, 1].astype(float)       # prompts (base; × social in loop)
item_shift_share   = 0.08 * item_popularity + 0.05 * item_freshness  # popular/fresh → more shares
item_shift_buy     = 0.20 * item_ct_onehot[:, 2].astype(float)       # commerce → purchases
item_shift_view    = 0.02 * item_popularity                          # popular → more clicks

# Zero-center all shifts so they don't shift the overall rate
for s in [item_shift_like, item_shift_long, item_shift_fav,
          item_shift_comment, item_shift_share, item_shift_buy, item_shift_view]:
    s -= s.mean()

# Behavior parameters: (b_threshold, shift_array, commerce_only)
# Tuned to hit: view~68%, like~22%, long_view~22%, fav~10%, comment~4%, share~5%, buy~1%
BEHAVIOR_SPEC = {
    'view':       (1.1,  item_shift_view,    False),
    'like':       (3.4,  item_shift_like,    False),
    'long_view':  (3.4,  item_shift_long,    False),
    'fav':        (4.4,  item_shift_fav,     False),
    'comment':    (5.4,  item_shift_comment, False),  # shift × social_tendency in loop
    'share':      (5.1,  item_shift_share,   False),
    'buy':        (6.9,  item_shift_buy,     True),   # commerce items only
}

print('Score + shift functions ready (Plan C)')
print(f'A_COMMON={A_COMMON}, behaviors: {list(BEHAVIOR_SPEC.keys())}')
print(f'Per-item shift magnitudes (std):')
for name, (b, s, _) in BEHAVIOR_SPEC.items():
    print(f'  {name:12s}: b={b:.1f}, shift std={s.std():.4f}, range=[{s.min():+.3f}, {s.max():+.3f}]')

---
## 5. 生成 Enriched Interactions

8% 热门（幂律） + 25% 同簇共享（pool=250） + 40% 全局池（pool=80，所有用户可见） + 27% 原交互 item → 行为漏斗 → 时间戳

In [ ]:
HOT_FRAC, CLUSTER_FRAC, GLOBAL_FRAC = 0.08, 0.25, 0.40
GLOBAL_POOL_SIZE = 80

# Global item pool (items everyone sees — guarantees base Jaccard)
global_pool = rng.choice(item_ids_arr, size=GLOBAL_POOL_SIZE, replace=False)

print('Generating enriched interactions (Plan C: A_COMMON + per-item shifts)...')
records = []
for i, uid in enumerate(user_latent.index):
    if uid not in user_sessions: continue
    user_vec = user_latent.loc[uid]
    uc = user_latent.loc[uid, 'cluster']
    social = float(user_vec['social_tendency'])
    user_orig = interactions[interactions['user_id'] == uid]['item_id'].values

    for session_base, n_events in user_sessions[uid]:
        session_items = []
        n_global = int(n_events * GLOBAL_FRAC)
        session_items.extend(rng.choice(global_pool, size=n_global, replace=True))
        n_hot = int(n_events * HOT_FRAC)
        session_items.extend(item_ids_arr[rng.choice(n_items, size=n_hot, p=item_probs, replace=True)])
        n_cluster = int(n_events * CLUSTER_FRAC)
        cpool = list(cluster_item_pool[uc])
        session_items.extend(rng.choice(cpool, size=n_cluster, replace=True))
        n_orig = n_events - n_global - n_hot - n_cluster
        if len(user_orig) > 0:
            session_items.extend(rng.choice(user_orig, size=n_orig, replace=True))
        else:
            session_items.extend(rng.choice(item_ids_arr, size=n_orig, replace=True))
        rng.shuffle(session_items)

        et = [session_base]
        for _ in range(n_events - 1):
            et.append(et[-1] + np.timedelta64(rng.randint(30, 300), 's'))

        for item_id, evt in zip(session_items, et):
            if item_id not in item_idx_map: continue
            idx = item_idx_map[item_id]
            raw_score = compute_raw(user_vec, idx)
            score = float(sigmoid(8.0 * raw_score))  # match_score (same as compute_score)
            rec = {'user_id': uid, 'item_id': item_id, 'match_score': score,
                   'event_time_ms': int(evt.astype('datetime64[ms]').astype(np.int64)),
                   'cluster': int(uc)}
            is_c = item_is_commerce[idx]

            for behavior, (b, shift_arr, commerce_only) in BEHAVIOR_SPEC.items():
                if commerce_only and not is_c:
                    rec[behavior] = 0; continue
                s = shift_arr[idx]
                if behavior == 'comment':
                    s = s * social  # user social tendency modulates comment probability
                p = sigmoid(A_COMMON * (raw_score + s) - b)
                rec[behavior] = int(rng.random() < p)
            records.append(rec)

    if (i + 1) % 1000 == 0:
        print(f'  {i+1}/{len(user_latent)} users ({len(records):,} records)')

interactions_enriched = pd.DataFrame(records)
print(f'Generated {len(interactions_enriched):,} interactions')
for col in ['view','like','long_view','fav','comment','share','buy']:
    print(f'  {col}: {interactions_enriched[col].mean()*100:.1f}%')

---
## 6. 验证

In [ ]:
print('=== Validation ===')
times = pd.to_datetime(interactions_enriched['event_time_ms'], unit='ms')
print(f'Time range: {times.min()} ~ {times.max()}')

iu_new = interactions_enriched.groupby('user_id').size()
print(f'Interactions/user: mean={iu_new.mean():.1f}')

# Jaccard
user_items = defaultdict(set)
for uid, iid in zip(interactions_enriched['user_id'], interactions_enriched['item_id']):
    user_items[uid].add(iid)
su = np.random.choice(list(user_items.keys()), size=min(500, len(user_items)), replace=False)
jaccards = []
for a in range(len(su)):
    for b in range(a+1, min(a+50, len(su))):
        s1, s2 = user_items[su[a]], user_items[su[b]]
        u = len(s1 | s2)
        if u > 0: jaccards.append(len(s1 & s2) / u)
ja = np.array(jaccards)
print(f'Jaccard: mean={ja.mean():.4f}, >5%={(ja>0.05).mean()*100:.1f}%')

# Top item share
item_freq = interactions_enriched['item_id'].value_counts()
top5_share = item_freq.head(int(n_items*0.05)).sum() / len(interactions_enriched)
print(f'Top 5% item share: {top5_share*100:.1f}%')

# Sessions
sc = interactions_enriched.groupby('user_id')['event_time_ms'].apply(
    lambda x: (x.sort_values().diff() > 3_600_000).sum() + 1)
print(f'Sessions/user: mean={sc.mean():.1f}')

# Behavior rates (Plan C: long_view is independent)
behaviors = ['view','like','long_view','fav','comment','share','buy']
print(f'\nBehavior rates:')
for b in behaviors:
    print(f'  {b}: {interactions_enriched[b].mean()*100:.1f}%')
eng = (interactions_enriched[['like','long_view','fav','comment','share']].max(axis=1))
strong = (interactions_enriched[['like','fav','comment','share']].max(axis=1))
print(f'  engagement (any of like/long/fav/comment/share): {eng.mean()*100:.1f}%')
print(f'  strong_action (any of like/fav/comment/share): {strong.mean()*100:.1f}%')
print(f'  engagement != strong_action: {(eng != strong).mean()*100:.1f}% (long_view contributes this)')

# GBDT quick check: match_score → engagement AUC
from sklearn.metrics import roc_auc_score
print(f'\nAUC (match_score → behavior):')
for b in behaviors:
    if interactions_enriched[b].nunique() > 1:
        auc = roc_auc_score(interactions_enriched[b], interactions_enriched['match_score'])
        print(f'  ms→{b}: {auc:.4f}')
print(f'  ms→engagement: {roc_auc_score(eng, interactions_enriched["match_score"]):.4f}')
print(f'  ms→strong_action: {roc_auc_score(strong, interactions_enriched["match_score"]):.4f}')

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes[0,0].hist(times, bins=30, color='#4ECDC4', alpha=0.7)
axes[0,0].set_title('Events Over Time'); axes[0,0].set_xlabel('Date')
axes[0,1].loglog(np.arange(1, len(item_freq)+1), item_freq.values, linewidth=1, color='#FF6B6B')
axes[0,1].set_title('Item Popularity (log-log)')
axes[0,2].hist(iu_new, bins=50, color='#FFE66D', alpha=0.7)
axes[0,2].axvline(iu_new.mean(), color='#333', linestyle='--', label=f'Mean: {iu_new.mean():.0f}')
axes[0,2].set_title('Interactions per User'); axes[0,2].legend()
axes[1,0].hist(ja, bins=50, color='#4ECDC4', alpha=0.7)
axes[1,0].axvline(0.05, color='#FF6B6B', linestyle='--', label='Target: 5%')
axes[1,0].set_title('User-User Jaccard'); axes[1,0].legend()
bcols = ['view','like','long_view','fav','comment','share','buy']
br = [interactions_enriched[c].mean()*100 for c in bcols]
axes[1,1].bar(bcols, br, color='#4ECDC4')
for k, v in enumerate(br): axes[1,1].text(k, v+1, f'{v:.1f}%', ha='center', fontsize=8)
axes[1,1].set_title('Behavior Rates (Plan C)')
axes[1,1].tick_params(axis='x', rotation=30)
axes[1,2].hist(sc, bins=20, color='#FF6B6B', alpha=0.7)
axes[1,2].set_title('Sessions per User (gap>1h)')
fig.tight_layout(); plt.show()

---
## 7. 保存

In [ ]:
# ============================================================
# Plan C: Compute interaction features for user/item enrichment
# ============================================================
# These features provide additional signal for the model to learn from:
# per-user behavior rates, per-item engagement stats, etc.

print('Computing interaction features...')

# --- Per-user interaction stats ---
user_stats = interactions_enriched.groupby('user_id').agg(
    user_n_interact=('item_id', 'size'),
    user_avg_ms=('match_score', 'mean'),
    user_std_ms=('match_score', 'std'),
    user_like_rate=('like', 'mean'),
    user_long_rate=('long_view', 'mean'),
    user_fav_rate=('fav', 'mean'),
    user_comment_rate=('comment', 'mean'),
    user_share_rate=('share', 'mean'),
    user_buy_rate=('buy', 'mean'),
).fillna(0)

# --- Per-item interaction stats ---
item_stats = interactions_enriched.groupby('item_id').agg(
    item_n_interact=('user_id', 'size'),
    item_avg_ms=('match_score', 'mean'),
    item_like_rate=('like', 'mean'),
    item_long_rate=('long_view', 'mean'),
    item_fav_rate=('fav', 'mean'),
    item_comment_rate=('comment', 'mean'),
    item_share_rate=('share', 'mean'),
    item_buy_rate=('buy', 'mean'),
).fillna(0)

# Merge into latent DataFrames
user_latent = user_latent.join(user_stats, how='left').fillna(0)
item_latent = item_latent.join(item_stats, how='left').fillna(0)

print(f'User features: {len(user_latent.columns)} columns (added {len(user_stats.columns)} interaction features)')
print(f'Item features: {len(item_latent.columns)} columns (added {len(item_stats.columns)} interaction features)')
print(f'  user_n_interact: mean={user_stats["user_n_interact"].mean():.0f}, max={user_stats["user_n_interact"].max():.0f}')
print(f'  item_n_interact: mean={item_stats["item_n_interact"].mean():.0f}, max={item_stats["item_n_interact"].max():.0f}')

# --- Save ---
interactions_enriched.to_csv(f'{DATA_DIR}/interactions_enriched.csv', index=False)
user_latent.to_csv(f'{DATA_DIR}/user_latent.csv', index_label='user_id')
item_latent.to_csv(f'{DATA_DIR}/item_latent.csv', index_label='item_id')
user_latent[['cluster']].to_csv(f'{DATA_DIR}/user_clusters.csv')

print(f'Saved {len(interactions_enriched):,} interactions to {DATA_DIR}/interactions_enriched.csv')
print(f'Saved enriched user_latent ({len(user_latent.columns)} cols) to {DATA_DIR}/user_latent.csv')
print(f'Saved enriched item_latent ({len(item_latent.columns)} cols) to {DATA_DIR}/item_latent.csv')
print(f'Saved user clusters to {DATA_DIR}/user_clusters.csv')
print()
print('=== Phase B Summary ===')
print(f'Time range: {times.min()} ~ {times.max()}')
print(f'Interactions: {len(interactions_enriched):,}')
print(f'Users: {interactions_enriched["user_id"].nunique():,}')
print(f'Items touched: {interactions_enriched["item_id"].nunique():,}')
print(f'Interactions/user: {iu_new.mean():.1f}')
print(f'Sessions/user: {sc.mean():.1f}')
print(f'Jaccard: {ja.mean():.4f} (>5%: {(ja>0.05).mean()*100:.1f}%)')
print(f'Top 5% item share: {top5_share*100:.1f}%')
print(f'Interaction features added: {len(user_stats.columns)} user + {len(item_stats.columns)} item')
print('Next: Phase C — KuaiRand 7 files (use long_view from Phase B directly)')